# Phase 2 — Time Horizon and Discounting

**Goal:** Relax three assumptions — A1 (proportional costs), A2 (one-year horizon), and A3 (no decay) — and show that time-adjusted rankings differ materially from the Phase 1 baseline.

## Assumptions relaxed

- **A1 (proportional costs):** Phase 1 assumed costs scale proportionally with beneficiaries. We now decompose costs into fixed ($C_i^{fixed}$) and variable ($C_i^{var}$) components, and adjust for compliance rate $p_c$.
- **A2 (one-year horizon):** Phase 1 assumed all interventions produce effects over exactly one year. We now use heterogeneous durations $T_i$.
- **A3 (no decay):** Phase 1 assumed effects persist at full strength indefinitely. We now model exponential decay with annual retention rate $\rho_i$.

## Key formulas

**Discounted effective effect:**
$$\hat{\tau}_i^{eff} = \hat{\tau}_i \cdot \sum_{t=0}^{T_i - 1} \frac{\rho_i^t}{(1+r)^t}$$

**Adjusted cost:**
$$C_i^{adj} = \frac{C_i^{fixed}}{N} + \frac{C_i^{var}}{p_{c,i}}$$

**Time-adjusted effectiveness:**
$$v_i^{adj} = \frac{\hat{\tau}_i^{eff}}{C_i^{adj}}$$

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cea_dev.core import InterventionSet
from cea_dev.time_horizon import DiscountedInterventionSet
from cea_dev.visualization import plot_league_table, plot_ranking_shift

pd.set_option("display.precision", 4)

## Load data and build Phase 1 baseline

In [ ]:
DATA_DIR = Path("../data")
iset = InterventionSet.from_csv(str(DATA_DIR / "interventions.csv"))
p1_table = iset.league_table()
print("Phase 1 — Unadjusted league table:")
p1_table

## Build Phase 2 — Time-adjusted league table

Using $N = 1{,}000$ beneficiaries (held constant across interventions for comparability).

In [ ]:
diset = DiscountedInterventionSet.from_intervention_set(iset)
p2_table = diset.league_table()
print("Phase 2 — Time-adjusted league table:")
p2_table

## Side-by-side comparison

**Headline result:** ECD moves from rank 5 to rank 2 when its 10-year duration is accounted for. Short-duration interventions with low retention (school meals, deworming) lose ground.

In [ ]:
shift = diset.ranking_shift(iset)
shift

In [ ]:
fig, (ax1, ax2) = plot_ranking_shift(shift)
plt.show()

## Sensitivity to discount rate

The plan specifies testing ECD's ranking robustness to discount rates in the range 2–6%. Here we check whether the broad pattern holds.

(Full sensitivity analysis is developed in Phase 4.)

In [ ]:
rates = [0.02, 0.03, 0.04, 0.05, 0.06]
for r in rates:
    temp_ivs = []
    for iv in iset.interventions:
        import copy
        temp = copy.deepcopy(iv)
        temp.discount_rate = r
        temp_ivs.append(temp)
    temp_iset = InterventionSet(temp_ivs)
    temp_diset = DiscountedInterventionSet.from_intervention_set(temp_iset)
    table = temp_diset.league_table()
    ecd_row = table[table["name"] == "Early childhood development (ECD)"]
    print(f"r = {r:.2f}: ECD rank = {ecd_row.index[0]}, CE* = ${ecd_row['ce_display'].values[0]:.2f}")

## Notes

1. **ECD moves from 5th to 2nd.** Its 10-year duration and 90% annual retention produce a cumulative effect multiplier of ~6.0×, more than offsetting its higher per-beneficiary cost. At $r = 3\%$, ECD's time-adjusted effectiveness is ~5× its Phase 1 value.
2. **Deworming and school meals lose ground.** Both have single-year durations and steep decay, so they receive no benefit from the time adjustment while their adjusted costs increase slightly (fixed costs spread per beneficiary).
3. **TaRL holds first place.** Its 2-year duration and 80% retention give it a ~1.8× effect multiplier, and its costs remain low. It is the most robust intervention in the set.
4. **Rank shifts are sensitive to discount rate.** ECD's rank advantage narrows at higher discount rates because its long-duration payoff is discounted more heavily. But even at $r = 6\%$, it remains in the top 3.
5. **Fixed costs matter at modest scale.** At $N = 1{,}000$, CAL's high fixed cost ($3{,}000) adds $3/beneficiary to its cost, noticeably degrading its CE ratio. At larger scale the fixed cost dilutes; at smaller scale it dominates. The choice of $N$ is therefore a sensitivity parameter (explored in Phase 4).